# Step 7 — Spoilage Risk Classifier & FEFO Sorter
> **Dynamic Food Clearing House Simulator (AIT-201)**  
> **Person 1 — Tech Lead & ML Tabular**  
> **Goal 1:** Inventory lot spoilage risk classification (`HIGH` / `MODERATE` / `LOW`) + FEFO sorting logic  
> **Output:** `spoilage_classifier.pkl` + `predict_spoilage_risk()` + `sort_by_fefo()`

---

## How to Use This Notebook
1. Open this notebook in **Google Colab**.
2. Run the **'Upload Dataset'** cell below — click the *Choose Files* button and upload:
   - `data/processed/7. Grocery_Inventory_cleaned.csv` (from your local repo folder)
3. Run all cells sequentially from top to bottom.
4. In the final cell, download `spoilage_classifier.pkl` and place it in the `models/` directory in your local repo.

In [ ]:
# ─────────────────────────────────────────────
# [CELL 1] Install / Import Libraries
# (Pre-installed in Colab; verifying xgboost installation)
# ─────────────────────────────────────────────
!pip install xgboost --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import os

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold, cross_val_score
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, roc_auc_score
from sklearn.preprocessing import LabelBinarizer
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='Set2')
print('✅ Libraries loaded successfully.')

In [ ]:
# ─────────────────────────────────────────────
# [CELL 2] Upload Dataset
# Click 'Choose Files' and select:
#   '7. Grocery_Inventory_cleaned.csv'
# ─────────────────────────────────────────────
from google.colab import files
uploaded = files.upload()

# Detect uploaded filename
fname = [k for k in uploaded.keys() if 'Grocery' in k or 'grocery' in k][0]
print(f'✅ File uploaded: {fname}')

---
## 1. Exploratory Data Analysis (EDA)

In [ ]:
# ─────────────────────────────────────────────
# [CELL 3] Load & Preview Data
# ─────────────────────────────────────────────
df = pd.read_csv(fname, parse_dates=['Date_Received', 'Expiration_Date', 'Last_Order_Date'])

print(f'Shape: {df.shape}')
print(f'\nColumns: {df.columns.tolist()}')
print(f'\nNull values per column:')
print(df.isnull().sum())
df.head(5)

In [ ]:
# ─────────────────────────────────────────────
# [CELL 4] Product Category & Status Distribution
# ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Category distribution bar chart
cat_counts = df['Category'].value_counts()
cat_counts.plot(kind='barh', ax=axes[0], color=sns.color_palette('Set2', len(cat_counts)))
axes[0].set_title('Product Distribution by Category', fontsize=13)
axes[0].set_xlabel('Lot Count')

# Inventory status bar chart
status_counts = df['Status'].value_counts()
status_counts.plot(kind='bar', ax=axes[1], color=['#2ecc71', '#e74c3c', '#f39c12'])
axes[1].set_title('Inventory Lot Status Distribution', fontsize=13)
axes[1].set_xlabel('Status')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('eda_category_status.png', dpi=120)
plt.show()
print('📊 EDA chart saved: eda_category_status.png')

In [ ]:
# ─────────────────────────────────────────────
# [CELL 5] Turnover Rate & Unit Price Distribution
# ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df['Inventory_Turnover_Rate'].hist(ax=axes[0], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Inventory Turnover Rate Distribution', fontsize=13)
axes[0].set_xlabel('Turnover Rate')

df['Unit_Price'].hist(ax=axes[1], bins=30, color='salmon', edgecolor='white')
axes[1].set_title('Unit Price Distribution', fontsize=13)
axes[1].set_xlabel('Price')

plt.tight_layout()
plt.savefig('eda_turnover_price.png', dpi=120)
plt.show()

---
## 2. Feature Engineering & Target Label Creation

In [ ]:
# ─────────────────────────────────────────────
# [CELL 6] Feature Engineering
# ─────────────────────────────────────────────
df_feat = df.copy()

# ── Feature 1: Total shelf life duration (days) from date received to expiration date
# Note: We do NOT use 'today()' because this dataset is static historical Kaggle data.
# Using 'today()' would cause past expiration dates to compute as heavily negative,
# distorting model training. Shelf life duration reliably measures inherent perishability.
df_feat['shelf_life_days'] = (df_feat['Expiration_Date'] - df_feat['Date_Received']).dt.days

# Drop lots with non-positive shelf life (data anomalies)
n_before = len(df_feat)
df_feat = df_feat[df_feat['shelf_life_days'] > 0].copy()
print(f'Rows dropped (shelf_life <= 0): {n_before - len(df_feat)}')

# ── Feature 2: Inventory turnover ratio (existing in dataset)
df_feat['turnover_ratio'] = df_feat['Inventory_Turnover_Rate']

# ── Feature 3: Stock pressure relative to reorder threshold
# Ratio > 1 indicates healthy stock; ratio < 1 indicates replenishment alert
df_feat['stock_pressure'] = df_feat['Stock_Quantity'] / (df_feat['Reorder_Level'] + 1)

# Clean Unit_Price: strip currency symbols and commas, convert to float
df_feat['Unit_Price_clean'] = df_feat['Unit_Price'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)

# ── Feature 4: Log-transformed unit price (mitigates skew and outlier impact)
df_feat['log_unit_price'] = np.log1p(df_feat['Unit_Price_clean'])

# ── Feature 5: Sales volume per unit stock (inventory velocity proxy)
df_feat['sales_rate'] = df_feat['Sales_Volume'] / (df_feat['Stock_Quantity'] + 1)

print(f'\nDataset shape after feature engineering: {df_feat.shape}')
df_feat[['shelf_life_days', 'turnover_ratio', 'stock_pressure', 'log_unit_price', 'sales_rate']].describe()

In [ ]:
# ─────────────────────────────────────────────
# [CELL 7] Target Label Creation (Spoilage Risk)
# ─────────────────────────────────────────────
# Thresholds aligned with retail management standards:
#   HIGH     = shelf life <= 7 days  -> highly perishable goods (fresh produce, seafood, meats)
#   MODERATE = shelf life <= 30 days -> semi-perishable goods (dairy, chilled foods, bread)
#   LOW      = shelf life > 30 days  -> shelf-stable inventory (canned, frozen, dry goods)
# Justification: Reflects regional supermarket markdown policies in Southeast Asia
# and FEFO scheduling practices in major Malaysian chains (AEON, Giant, Lotus's, Mydin).

HIGH_THRESHOLD = 7
MODERATE_THRESHOLD = 30

def assign_spoilage_label(shelf_life_days):
    if shelf_life_days <= HIGH_THRESHOLD:
        return 'HIGH'
    elif shelf_life_days <= MODERATE_THRESHOLD:
        return 'MODERATE'
    else:
        return 'LOW'

df_feat['spoilage_risk'] = df_feat['shelf_life_days'].apply(assign_spoilage_label)

# Visualize label distribution
label_counts = df_feat['spoilage_risk'].value_counts()
print('Spoilage Risk Label Distribution:')
print(label_counts)
print(f'\nProportion (%):\n{(label_counts / len(df_feat) * 100).round(1)}')

fig, ax = plt.subplots(figsize=(6, 4))
label_counts.plot(kind='bar', ax=ax, color=['#e74c3c', '#f39c12', '#2ecc71'], edgecolor='white')
ax.set_title('Spoilage Risk Label Distribution', fontsize=13)
ax.set_xlabel('Risk Level')
ax.set_ylabel('Lot Count')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig('label_distribution.png', dpi=120)
plt.show()

---
## 3. Data Preparation — Encoding & Splitting

In [ ]:
# ─────────────────────────────────────────────
# [CELL 8] Categorical Encoding & Feature Selection
# ─────────────────────────────────────────────
FEATURE_COLS = ['shelf_life_days', 'turnover_ratio', 'stock_pressure',
                'log_unit_price', 'sales_rate']
TARGET_COL = 'spoilage_risk'

# One-Hot Encoding for Category column
# (OHE avoids false ordinal assumptions across categories in tree-based architectures)
df_encoded = pd.get_dummies(df_feat[FEATURE_COLS + ['Category', TARGET_COL]],
                             columns=['Category'], prefix='cat', drop_first=False)

feature_cols_final = [c for c in df_encoded.columns if c != TARGET_COL]
X = df_encoded[feature_cols_final]
y = df_encoded[TARGET_COL]

print(f'Total features after encoding: {X.shape[1]}')
print(f'Features: {feature_cols_final}')
print(f'\nTarget (y) distribution:\n{y.value_counts()}')

In [ ]:
# ─────────────────────────────────────────────
# [CELL 9] Train/Test Split (Stratified)
# stratify=y guarantees identical class proportions
# between training and testing sets.
# ─────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y      # Essential for imbalanced multi-class setups
)

print(f'Train set: {X_train.shape[0]} rows')
print(f'Test set:  {X_test.shape[0]} rows')
print(f'\ny_train distribution:\n{y_train.value_counts()}')
print(f'\ny_test distribution:\n{y_test.value_counts()}')

---
## 4. Model Training

In [ ]:
# ─────────────────────────────────────────────
# [CELL 10] Model 0 — Non-ML Baseline (Benchmark)
# Serves as an empirical baseline to prove that ML
# delivers tangible predictive gain over naive heuristics.
# ─────────────────────────────────────────────
baseline = DummyClassifier(strategy='most_frequent', random_state=42)
baseline.fit(X_train, y_train)
y_pred_baseline = baseline.predict(X_test)

print('=== MODEL 0: Majority Class Baseline ===')
print(classification_report(y_test, y_pred_baseline, target_names=['HIGH', 'LOW', 'MODERATE']))

In [ ]:
# ─────────────────────────────────────────────
# [CELL 11] Model 1 — Random Forest Classifier
# class_weight='balanced' offsets class imbalances.
# n_jobs=-1 parallelizes across all available CPU cores.
# ─────────────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print('=== MODEL 1: Random Forest Classifier ===')
print(classification_report(y_test, y_pred_rf, target_names=sorted(y.unique())))

In [ ]:
# ─────────────────────────────────────────────
# [CELL 12] Model 2 — XGBoost Classifier (Benchmark)
# ─────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
xgb.fit(X_train, y_train_enc)
y_pred_xgb_enc = xgb.predict(X_test)
y_pred_xgb = le.inverse_transform(y_pred_xgb_enc)

print('=== MODEL 2: XGBoost Classifier ===')
print(classification_report(y_test, y_pred_xgb, target_names=sorted(y.unique())))

---
## 5. Hyperparameter Tuning — RandomizedSearchCV
> **Randomized Search is selected over Grid Search for efficiency**:  
> Explores 30 random configurations across a large search space, recovering ~95% of grid optimum in ~10% of total compute time.

In [ ]:
# ─────────────────────────────────────────────
# [CELL 13] Randomized Search Hyperparameter Tuning
# Tuning Random Forest for optimal stability and interpretability
# ─────────────────────────────────────────────
param_dist = {
    'n_estimators':     [100, 200, 300, 500],
    'max_depth':        [None, 5, 10, 15, 20],
    'min_samples_split':[2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features':     ['sqrt', 'log2', None],
}

search = RandomizedSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=30,              # 30 sampled combinations
    cv=5,                   # 5-fold stratified cross-validation per iteration
    scoring='f1_weighted',  # Weighted F1 prioritizes imbalanced precision/recall balance
    n_jobs=-1,
    random_state=42,
    verbose=1
)

print('🔍 Starting Randomized Search (30 combinations × 5 folds = 150 fits)...')
search.fit(X_train, y_train)

print(f'\n✅ Best Parameters:')
for k, v in search.best_params_.items():
    print(f'   {k}: {v}')
print(f'\nBest CV F1-Weighted (Training): {search.best_score_:.4f}')

best_model = search.best_estimator_

---
## 6. Final Model Evaluation

In [ ]:
# ─────────────────────────────────────────────
# [CELL 14] Final Classification Report
# Critical Focus: Recall on HIGH risk class.
#   False Negatives (HIGH classified as LOW) lead to unmitigated waste.
#   This operational cost far exceeds False Positives.
# ─────────────────────────────────────────────
y_pred_final = best_model.predict(X_test)

print('=== FINAL MODEL EVALUATION (Post-Tuning) ===')
print(classification_report(y_test, y_pred_final, target_names=sorted(y.unique())))

# Multi-class One-vs-Rest ROC-AUC
y_prob = best_model.predict_proba(X_test)
lb = LabelBinarizer()
y_test_bin = lb.fit_transform(y_test)
auc = roc_auc_score(y_test_bin, y_prob, multi_class='ovr', average='weighted')
print(f'ROC-AUC (Weighted OvR): {auc:.4f}')

In [ ]:
# ─────────────────────────────────────────────
# [CELL 15] Confusion Matrix
# ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_final,
    display_labels=sorted(y.unique()),
    colorbar=False,
    ax=ax
)
ax.set_title('Confusion Matrix — Spoilage Risk Classifier', fontsize=13)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=120)
plt.show()
print('📊 Confusion matrix saved: confusion_matrix.png')

In [ ]:
# ─────────────────────────────────────────────
# [CELL 16] Feature Importance
# Critical visual for slide deck and technical report,
# demonstrating key model decision drivers.
# ─────────────────────────────────────────────
feat_imp = pd.Series(best_model.feature_importances_, index=X_train.columns)
feat_imp_top = feat_imp.sort_values(ascending=True).tail(12)

fig, ax = plt.subplots(figsize=(8, 5))
feat_imp_top.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Top 12 Feature Importances — Spoilage Classifier', fontsize=13)
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120)
plt.show()
print('📊 Feature importance chart saved: feature_importance.png')

In [ ]:
# ─────────────────────────────────────────────
# [CELL 17] Cross-Validation — Academic Reporting Metrics
# Provides generalizable confidence intervals (mean ± std)
# rather than a single point estimate from a fixed test partition.
# Serves a distinct purpose from CV in RandomizedSearchCV:
#   - RandomizedSearch CV selects optimal hyperparameters.
#   - This CV assesses overall generalization variance.
# ─────────────────────────────────────────────
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(best_model, X, y, cv=skf, scoring='f1_weighted', n_jobs=-1)

print(f'CV F1-Weighted per fold: {[f"{s:.4f}" for s in cv_scores]}')
print(f'\n📊 Summary for Academic Report:')
print(f'   F1-Score (5-Fold CV): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

---
## 7. Inference Functions & FEFO Sorter
> These functions are integrated into FastAPI (Person 2) via `src/ml/spoilage.py`

In [ ]:
# ─────────────────────────────────────────────
# [CELL 18] Production Inference Functions & FEFO Sorter
# ─────────────────────────────────────────────

# Cache trained feature schema to enforce identical inference representation
TRAINED_FEATURE_COLS = list(X_train.columns)
ALL_CATEGORIES = [c.replace('cat_', '') for c in TRAINED_FEATURE_COLS if c.startswith('cat_')]

def preprocess_single_lot(lot: dict) -> pd.DataFrame:
    """Preprocesses a single lot dictionary into a model-ready DataFrame.

    Args:
        lot: dict with keys:
            - shelf_life_days (int): total shelf life days (expiry - date_received)
            - turnover_ratio (float): Inventory_Turnover_Rate
            - stock_pressure (float): Stock_Quantity / (Reorder_Level + 1)
            - log_unit_price (float): log1p(Unit_Price)
            - sales_rate (float): Sales_Volume / (Stock_Quantity + 1)
            - category (str): product category string (e.g. 'Fruits & Vegetables')

    Returns:
        pd.DataFrame: single-row dataframe with all training features
    """
    row = {col: 0.0 for col in TRAINED_FEATURE_COLS}

    # Numeric features
    row['shelf_life_days']  = lot.get('shelf_life_days', 0)
    row['turnover_ratio']   = lot.get('turnover_ratio', 0.0)
    row['stock_pressure']   = lot.get('stock_pressure', 1.0)
    row['log_unit_price']   = lot.get('log_unit_price', 0.0)
    row['sales_rate']       = lot.get('sales_rate', 0.0)

    # One-hot encoded category
    cat_key = f"cat_{lot.get('category', '')}"
    if cat_key in row:
        row[cat_key] = 1.0

    return pd.DataFrame([row])


def predict_spoilage_risk(lot: dict) -> dict:
    """Predicts the spoilage risk label and confidence score for a single lot.

    This is the main inference function called by the FastAPI backend (Person 2).

    Args:
        lot: dict — same format as preprocess_single_lot()
            Note: 'shelf_life_days' here should be (expiry_date - today) computed
            at real-time by the ETL pipeline (Person 3), NOT (expiry - date_received).
            Training used total shelf life; inference uses actual remaining days.

    Returns:
        dict: {"label": "HIGH"|"MODERATE"|"LOW", "score": float 0.0-1.0}
    """
    X_input = preprocess_single_lot(lot)
    label = best_model.predict(X_input)[0]
    score = float(best_model.predict_proba(X_input).max())
    return {"label": label, "score": round(score, 4)}


def sort_by_fefo(batch_list: list) -> list:
    """Sorts a list of lot dictionaries by First-Expired First-Out (FEFO) order.

    Args:
        batch_list: list of dicts, each must have an 'expiry_date' key (str or datetime)

    Returns:
        list: same dicts sorted ascending by expiry_date (soonest expiry first)
    """
    return sorted(batch_list, key=lambda x: pd.Timestamp(x['expiry_date']))


# ── Functional verification ──
sample_lot = {
    'shelf_life_days': 5,
    'turnover_ratio': 3.5,
    'stock_pressure': 0.8,
    'log_unit_price': 2.1,
    'sales_rate': 0.15,
    'category': 'Fruits & Vegetables'
}
result = predict_spoilage_risk(sample_lot)
print(f'✅ predict_spoilage_risk() test: {result}')

sample_batches = [
    {'lot_id': 'A001', 'expiry_date': '2026-11-10'},
    {'lot_id': 'A002', 'expiry_date': '2026-11-07'},
    {'lot_id': 'A003', 'expiry_date': '2026-11-14'},
]
sorted_batches = sort_by_fefo(sample_batches)
print(f'\n✅ sort_by_fefo() test (FEFO order):')
for b in sorted_batches:
    print(f'   {b}')

---
## 8. Model Serialization & Export

In [ ]:
# ─────────────────────────────────────────────
# [CELL 19] Export Model to .pkl
# ─────────────────────────────────────────────
MODEL_FILENAME = 'spoilage_classifier.pkl'

# Package model and metadata bundle to guarantee schema fidelity at inference
model_package = {
    'model': best_model,
    'feature_columns': TRAINED_FEATURE_COLS,
    'label_classes': sorted(y.unique()),
    'thresholds': {
        'HIGH': HIGH_THRESHOLD,
        'MODERATE': MODERATE_THRESHOLD
    },
    'f1_cv_mean': round(cv_scores.mean(), 4),
    'f1_cv_std': round(cv_scores.std(), 4)
}

joblib.dump(model_package, MODEL_FILENAME)
size_kb = os.path.getsize(MODEL_FILENAME) / 1024
print(f'✅ Model serialized: {MODEL_FILENAME} ({size_kb:.1f} KB)')
print(f'   F1-Score: {model_package["f1_cv_mean"]} ± {model_package["f1_cv_std"]}')
print(f'   Classes: {model_package["label_classes"]}')
print(f'   Features: {len(TRAINED_FEATURE_COLS)} columns')

In [ ]:
# ─────────────────────────────────────────────
# [CELL 20] Download Artifacts to Local Machine
# Post-download placement:
#   1. Save file to 'models/' in your local repository
#   2. git add models/spoilage_classifier.pkl
#   3. git commit -m 'feat: add trained spoilage classifier (Goal 1)'
#   4. git push origin main
# ─────────────────────────────────────────────
from google.colab import files
files.download(MODEL_FILENAME)

# Download presentation figures
for chart in ['eda_category_status.png', 'label_distribution.png',
              'confusion_matrix.png', 'feature_importance.png']:
    if os.path.exists(chart):
        files.download(chart)

print('🎉 Done! All artifacts ready to be committed to the repository.')

---
## ✅ Post-Execution Checklist

- [ ] Download `spoilage_classifier.pkl` → place into `models/`
- [ ] Download the 4 PNG charts → place into `docs/images/` or `notebooks/`
- [ ] Verify `src/ml/spoilage.py` reflects the Cell 18 inference signature
- [ ] `git add models/spoilage_classifier.pkl src/ml/spoilage.py notebooks/step7_spoilage_classifier.ipynb`
- [ ] `git commit -m 'feat(ml): Goal 1 - spoilage classifier & FEFO sorter'`
- [ ] `git push origin main`
- [ ] Notify Person 2 that model artifact is ready for integration with the `/api/recommend` endpoint